# TRACER — Module 5: Adversarial Detection Engine

Builds a real clean-vs-FGSM-vs-PGD dataset from Flickr8k images, extracts real
attention-entropy features from real CLIP, trains the detector, and reports real accuracy —
this is where TRACER's actual detection claim gets tested against data, not synthetic
stand-ins.

Self-contained — clones the repo automatically. Assumes Flickr8k is already in your Drive
from Module 3.

In [ ]:
GITHUB_USERNAME = "IqraYasmin123"   # change if this isn't your username
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/tracers.git"

import os
if not os.path.exists("/content/tracers"):
    !git clone {REPO_URL} /content/tracers
else:
    %cd /content/tracers
    !git pull

%cd /content/tracers/ai-engine
import sys
sys.path.insert(0, ".")

from src.vlm import CLIPVLM, VLMConfig
from src.dataset import FlickrDataset, DatasetConfig
from src.attacks import FGSMAttack, PGDAttack, AttackConfig
from src.detection import (
    EntropyClassifier, DetectionConfig, extract_attention_entropy,
    plot_entropy_profiles, plot_confusion_matrix, plot_roc_curve,
)
import torch
import numpy as np
print("Imports OK.")


## Load CLIP and Flickr8k

**Performance note:** Flickr8k is copied from Drive to Colab's local disk first. Google
Drive is a network-mounted filesystem in Colab — checking whether ~8,091 image files exist
directly against Drive is slow (thousands of small network round-trips). A one-time local
copy makes both the initial dataset load and every subsequent image read in this notebook
much faster, while the dataset itself still persists in Drive between sessions.

The copy is verified complete via a marker file, not just folder existence — if a previous
copy was interrupted (e.g. by restarting the runtime mid-copy), this correctly detects the
incomplete copy and redoes it from scratch, rather than silently using partial data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

vlm = CLIPVLM(VLMConfig())
print(f"CLIP loaded on {vlm.device}")

DRIVE_DATASET_DIR = "/content/drive/MyDrive/TRACER/datasets/flickr8k"
LOCAL_DATASET_DIR = "/content/flickr8k_local"
COPY_MARKER = os.path.join(LOCAL_DATASET_DIR, ".copy_complete")

if not os.path.exists(COPY_MARKER):
    print("Copying Flickr8k to local Colab disk (redoing from scratch if a previous "
          "attempt was interrupted)...")
    !rm -rf "{LOCAL_DATASET_DIR}"
    !cp -r "{DRIVE_DATASET_DIR}" "{LOCAL_DATASET_DIR}"
    with open(COPY_MARKER, "w") as f:
        f.write("done")
    print("Copy complete.")
else:
    print("Local copy already present and verified complete — skipping copy.")

dataset = FlickrDataset(DatasetConfig(root_dir=LOCAL_DATASET_DIR, image_size=(224, 224)))
print(f"Loaded {len(dataset)} (image, caption) pairs.")


## Build a real clean / FGSM / PGD dataset

For each of N images: encode the clean version, an FGSM-attacked version, and a
PGD-attacked version, extracting attention-entropy features for all three. This is the same
pattern used in the original FYP prototype notebook, now running through your production
Module 2-5 code instead of one-off script cells.

`N_SAMPLES = 80` keeps this runnable in a few minutes on a T4 — raise it later for your final
results chapter once everything below works correctly.

In [ ]:
N_SAMPLES = 80
fgsm_config = AttackConfig(epsilon=8/255)
pgd_config = AttackConfig(epsilon=8/255, alpha=2/255, steps=10)

features, binary_labels, attack_labels = [], [], []

for i in range(N_SAMPLES):
    sample = dataset[i]
    pixel_values = vlm.preprocess_image(sample["image"])
    text_embeds = vlm.encode_text([sample["caption"]])

    fgsm_adv = FGSMAttack(fgsm_config).generate(vlm, pixel_values, text_embeds)
    pgd_adv = PGDAttack(pgd_config).generate(vlm, pixel_values, text_embeds)

    for pv, is_adv, attack_name in [
        (pixel_values, 0, "clean"),
        (fgsm_adv, 1, "fgsm"),
        (pgd_adv, 1, "pgd"),
    ]:
        entropy = extract_attention_entropy(vlm, pv).cpu().numpy().flatten()
        features.append(entropy)
        binary_labels.append("adversarial" if is_adv else "clean")
        attack_labels.append(attack_name)

    if (i + 1) % 20 == 0:
        print(f"Processed {i + 1}/{N_SAMPLES} images...")

X = np.array(features)
y_binary = np.array(binary_labels)
y_attack = np.array(attack_labels)
print(f"\nBuilt {len(X)} feature vectors: {sum(y_binary=='clean')} clean, {sum(y_binary=='adversarial')} adversarial")


## Train and evaluate the binary detector (clean vs. adversarial)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.3, random_state=42, stratify=y_binary
)

detector = EntropyClassifier(DetectionConfig(max_iter=1000)).fit(X_train, y_train)
predictions = detector.predict(X_test)
probs = detector.predict_proba(X_test)

adv_index = list(detector.classes_).index("adversarial")
auc = roc_auc_score((y_test == "adversarial").astype(int), probs[:, adv_index])

print(classification_report(y_test, predictions))
print(f"AUC: {auc:.4f}")


## Visualize: entropy profile, confusion matrix, ROC curve

In [ ]:
fig1 = plot_entropy_profiles(X, y_binary, class_names=["clean", "adversarial"])


In [ ]:
fig2 = plot_confusion_matrix(y_test, predictions, labels=["clean", "adversarial"])


In [ ]:
fig3 = plot_roc_curve(y_test, probs[:, adv_index], positive_label="adversarial")


## Bonus: attack-type classification (clean vs. FGSM vs. PGD)

Same `EntropyClassifier` class, trained on the 3-class labels instead — demonstrating the
"one classifier, two roles" design from `docs/module5_detection.md`.

In [ ]:
Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X, y_attack, test_size=0.3, random_state=42, stratify=y_attack
)

attack_classifier = EntropyClassifier(DetectionConfig(max_iter=1000)).fit(Xa_train, ya_train)
attack_predictions = attack_classifier.predict(Xa_test)

print(classification_report(ya_test, attack_predictions))
fig4 = plot_confusion_matrix(ya_test, attack_predictions, labels=["clean", "fgsm", "pgd"])


## Save the trained detector weights (for Module 9's backend to load later)

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/TRACER/models", exist_ok=True)
detector.save("/content/drive/MyDrive/TRACER/models/entropy_detector.joblib")
attack_classifier.save("/content/drive/MyDrive/TRACER/models/attack_classifier.joblib")
print("Saved both models to Drive.")


## Module 5 completion check

Run the full test suite:
```bash
cd ai-engine
pytest tests/test_detection.py -v
```
All 12 tests should pass. Combined with the real accuracy/AUC numbers above (record these —
they go directly in your results chapter), Module 5 is complete. Continue to
**Module 6 — Attention Analysis & Heatmap Generation**.